# Spark Setup

In [1]:
import glob
import os
import shutil
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient, UpdateOne
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, expr, from_json, to_timestamp, abs as spark_abs,
    unix_timestamp, lit, to_date, sin, cos, sqrt, atan2, radians
)
from pyspark.sql.types import *
from datetime import datetime

HOST_IP = "192.168.64.1"
MONGO_URI = "mongodb://mongodb:27017/"
MONGO_DB = "fit3182_a2"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .config("spark.sql.shuffle.partitions", "5")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

print("Debug: SparkSession has been created successfully.")

def log_batch(name):

    def logger(batch_df, batch_id):

        now = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row_count = batch_df.count()

        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id}")
        print(f"Rows received: {row_count}")
        print("=" * 70)

        batch_df.show(
            truncate=False
        )

    return logger

Debug: SparkSession has been created successfully.


Accepting streams

In [2]:

# Create a JSON schema that matches the payload from producer for easier handling

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_camera_stream(topic, producer):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{HOST_IP}:9092")
        .option("subscribe", topic)
        .option("startingOffsets", "latest") # start from first batch
        .load()
        # The value from Kafka is in bytes, so we can cast it to a string
        .selectExpr("CAST(value AS STRING) as json_value")
        # Parse the string into the columns using the struct schema we defined
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert timestamp to proper Spark timestamp type
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag each event with its source
        .withColumn("source", lit(producer))
        .withWatermark("event_time", "10 minutes")
    )

camera_stream_a = read_camera_stream("camera-events-A", "1")
camera_stream_b = read_camera_stream("camera-events-B", "2")
camera_stream_c = read_camera_stream("camera-events-C", "3")

print("Debug: Kafka streams have been created for all three cameras.")

Debug: Kafka streams have been created for all three cameras.


join them with each other so easy process i guess idk ill figure out why later

In [3]:
camera_df = (
    spark.read.csv(f"{Path('..')}/data/camera.csv", header=True, inferSchema=True)
    .select("camera_id", "position", "speed_limit", "latitude", "longitude")
)

camera_df.show()
print(f"Debug: Camera loaded: {camera_df.count()} cameras.")

def join_stream_with_camera(stream):
    return stream.join(camera_df, on="camera_id", how="inner")

joined_stream_a = join_stream_with_camera(camera_stream_a)
joined_stream_b = join_stream_with_camera(camera_stream_b)
joined_stream_c = join_stream_with_camera(camera_stream_c)

+---------+--------+-----------+-----------+-----------+
|camera_id|position|speed_limit|   latitude|  longitude|
+---------+--------+-----------+-----------+-----------+
|        1|   152.5|        110|2.157730731|102.6601002|
|        2|   153.5|        110|2.162418757|102.6524549|
|        3|   154.5|         90|2.167352891|102.6449144|
+---------+--------+-----------+-----------+-----------+

Debug: Camera loaded: 3 cameras.


In [4]:
def get_instant_violations(stream):
    return (
        stream
        .filter(col("speed_reading") > col("speed_limit"))
        .withColumn("violation_type", lit("instantaneous"))
        .withColumn("violation_date", to_date(col("event_time")))
        .select(
            "car_plate",
            "batch_id",
            "violation_date",
            "violation_type",
            "camera_id",
            col("speed_reading").alias("speed_recorded"),
            "speed_limit",
            col("event_time").cast("string").alias("event_time"),
            "source" 
        )
    )

camera_a_instant_violations = get_instant_violations(joined_stream_a)
camera_b_instant_violations = get_instant_violations(joined_stream_b)
camera_c_instant_violations = get_instant_violations(joined_stream_c)

def write_json_per_batch(base_path):
    def _writer(batch_df, batch_id):
        # Append JSON Lines to a single file so batches accumulate.
        file_path = base_path
        if not file_path.endswith(".json"):
            file_path = f"{base_path}/results.json"
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        rows = batch_df.toJSON().collect()
        if not rows:
            return
        with open(file_path, "a", encoding="utf-8") as f:
            for row in rows:
                f.write(row + "\n")
    return _writer

camera_a_instant_query = (
    camera_a_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_a"))
    .start()
)

camera_b_instant_query = (
    camera_b_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_b"))
    .start()
)

camera_c_instant_query = (
    camera_c_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_c"))
    .start()
)

print("Debug: Instantaneous violations have been extracted and combined.")


Debug: Instantaneous violations have been extracted and combined.


average speed violations time baby

In [5]:
# A→B segment join (camera 1 to camera 2)
segment_ab = (
    joined_stream_a.alias("entry")
    .join(
        joined_stream_b.alias("exit"),
        expr("""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval 10 minutes
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source"),
        col("entry.latitude").alias("entry_latitude"),
        col("entry.longitude").alias("entry_longitude"),
        col("exit.latitude").alias("exit_latitude"),
        col("exit.longitude").alias("exit_longitude")
        
    )
)

# B→C segment join (camera 2 to camera 3)
segment_bc = (
    joined_stream_b.alias("entry")
    .join(
        joined_stream_c.alias("exit"),
        expr("""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval 10 minutes
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source"),
        col("entry.latitude").alias("entry_latitude"),
        col("entry.longitude").alias("entry_longitude"),
        col("exit.latitude").alias("exit_latitude"),
        col("exit.longitude").alias("exit_longitude")
    )
)

# segment_ab = (
#     segment_ab
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Segment A→B"))
#     .start()
# )

# segment_bc = (
#     segment_bc
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Segment B→C"))
#     .start()
# )

print("Debug: Segment joins have been defined for A→B and B→C.")

Debug: Segment joins have been defined for A→B and B→C.


In [6]:


def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers

    lat1_rad = radians(col(lat1))
    lon1_rad = radians(col(lon1))
    lat2_rad = radians(col(lat2))
    lon2_rad = radians(col(lon2))

    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad

    a = (
        sin(delta_lat / 2) * sin(delta_lat / 2)
        + cos(lat1_rad) * cos(lat2_rad) * sin(delta_lon / 2) * sin(delta_lon / 2)
    )
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance_km = R * c
    return distance_km

def compute_avg_speed(joined_segments):
    return (
        joined_segments
        .withColumn(
            "distance_km",
            calculate_haversine_distance(
                "entry_latitude", "entry_longitude",
                "exit_latitude", "exit_longitude"
            )
        )
        .withColumn(
            "travel_time_hours",
            (col("exit_time").cast("double") - col("entry_time").cast("double")) / 3600
                    )
        .withColumn(
            "average_speed",
            col("distance_km") / col("travel_time_hours")
        )
        .filter(col("average_speed") > col("speed_limit"))
        .withColumn("violation_type", lit("average"))
        .withColumn("violation_date", to_date(col("exit_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "end_camera_id",
            "average_speed",
            "speed_limit",
            col("exit_time").cast("string").alias("event_time"),
            "start_camera_id",
            "distance_km",
            "source",
            "entry_time",
            "exit_time",
            "entry_batch_id",
            "exit_batch_id"
    )
)


average_violations_ab = compute_avg_speed(segment_ab)
average_violations_bc = compute_avg_speed(segment_bc)

ab_avg_violations_query = (
    average_violations_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_ab"))
    .start()
)

bc_avg_violations_query = (
    average_violations_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_bc"))
    .start()
)

# avg_vio_ab_query = (
#     average_violations_ab
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations A→B"))
#     .start()
# )

# avg_vio_bc_query = (
#     average_violations_bc
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations B→C"))
#     .start()
# )


print("Average speed violation detection logic defined.")

Average speed violation detection logic defined.


## Task 2.1.3 — MongoDB Sink

Violations are written via `foreachBatch` using pymongo `bulk_write` with `UpdateOne` upserts.

**Daily merging (Task 2.1.4):** Multiple violations for the same `(car_plate, date, violation_type, camera_id)` 
are merged into one document. Each new incident is appended to an `incidents` array via `$push`, 
rather than creating a duplicate document. This matches the index defined in `mongo_setup.py` on `(car_plate, date)`.

**Idempotency:** Upserts are safe to retry — re-writing the same event just pushes a duplicate 
into the `incidents` array, which is preferable to crashing the stream.


In [7]:

def mongo_sink(name):
    def write_violations_to_mongo(batch_df, batch_id):
        rows = batch_df.collect()
        
        if not rows:
            print(f"Batch {batch_id} is empty, skipping MongoDB write. resolving: {name}")
            return

        operations = []
        for row in rows:
            doc = row.asDict()

            # Build the sub-document to push into the violations array
            if doc["violation_type"] == "instantaneous":
                violation_entry = {
                    "type":      "instant",
                    "camera_id": doc["camera_id"],
                    "speed":     doc["speed_recorded"],
                }
            else:
                violation_entry = {
                    "type":         "average",
                    "start_camera": doc["start_camera_id"],
                    "end_camera":   doc["end_camera_id"],
                    "avg_speed":    doc["average_speed"],
                }

            operations.append(
                UpdateOne(
                    # Match key — one document per car per day
                    {
                        "car_plate": doc["car_plate"],
                        "date": datetime.combine(doc["violation_date"], datetime.min.time()),
                    },
                    {
                        # $push always runs — appends to violations array on both
                        # insert and update, so same-day violations accumulate
                        "$push": {"violations": violation_entry},
                    },
                    upsert=True
                )
            )
            
        client = MongoClient(MONGO_URI)
        try:
            collection = client[MONGO_DB]["violations"]
            result = collection.bulk_write(operations, ordered=False)
            print(
                f"[Batch {batch_id}] Query {name}: {len(operations)} violation(s) written — "
                f"upserted: {result.upserted_count}, modified: {result.modified_count}"
            )
        except Exception as exc:
            print(f"[Batch {batch_id}] Query {name}: MongoDB write error: {exc}")
        finally:
            client.close()
    return write_violations_to_mongo

print("Debug: MongoDB sink function defined.")


Debug: MongoDB sink function defined.


In [8]:
camera_a_query_mongo = (
    camera_a_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_a_instant"))
    .start()
)

camera_b_query_mongo = (
    camera_b_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_b_instant"))
    .start()
)

camera_c_query_mongo = (
    camera_c_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_c_instant"))
    .start()
)

ab_avg_query_mongo = (
    average_violations_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("avg_ab"))
    .start()
)

bc_avg_query_mongo = (
    average_violations_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("avg_bc"))
    .start()
)

print("Debug: MongoDB streaming queries have been started for all violation types.")

Debug: MongoDB streaming queries have been started for all violation types.


In [ ]:
spark.streams.awaitAnyTermination()

Batch 0 is empty, skipping MongoDB write. resolving: camera_b_instant
Batch 0 is empty, skipping MongoDB write. resolving: camera_c_instant
Batch 0 is empty, skipping MongoDB write. resolving: camera_a_instant
Batch 0 is empty, skipping MongoDB write. resolving: avg_ab
Batch 0 is empty, skipping MongoDB write. resolving: avg_bc
[Batch 1] Query camera_a_instant: 11 violation(s) written — upserted: 11, modified: 0
[Batch 2] Query camera_a_instant: 22 violation(s) written — upserted: 22, modified: 0
Batch 1 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 1] Query camera_b_instant: 5 violation(s) written — upserted: 0, modified: 5
[Batch 3] Query camera_a_instant: 22 violation(s) written — upserted: 22, modified: 0
[Batch 2] Query camera_b_instant: 1 violation(s) written — upserted: 0, modified: 1
Batch 1 is empty, skipping MongoDB write. resolving: avg_bc
[Batch 4] Query camera_a_instant: 21 violation(s) written — upserted: 21, modified: 0
[Batch 3] Query camera_b_instant: 5 vi

[Batch 10] Query avg_bc: 5 violation(s) written — upserted: 0, modified: 5
[Batch 28] Query camera_b_instant: 10 violation(s) written — upserted: 0, modified: 10
Batch 24 is empty, skipping MongoDB write. resolving: camera_c_instant
[Batch 29] Query camera_a_instant: 7 violation(s) written — upserted: 7, modified: 0
[Batch 11] Query avg_ab: 8 violation(s) written — upserted: 0, modified: 8
Batch 29 is empty, skipping MongoDB write. resolving: camera_b_instant
Batch 25 is empty, skipping MongoDB write. resolving: camera_c_instant
[Batch 30] Query camera_a_instant: 20 violation(s) written — upserted: 20, modified: 0
[Batch 30] Query camera_b_instant: 5 violation(s) written — upserted: 0, modified: 5
[Batch 26] Query camera_c_instant: 2 violation(s) written — upserted: 0, modified: 2
[Batch 11] Query avg_bc: 3 violation(s) written — upserted: 0, modified: 3
[Batch 31] Query camera_a_instant: 19 violation(s) written — upserted: 19, modified: 0
[Batch 31] Query camera_b_instant: 1 violation

[Batch 52] Query camera_c_instant: 1 violation(s) written — upserted: 0, modified: 1
[Batch 55] Query camera_a_instant: 8 violation(s) written — upserted: 8, modified: 0
[Batch 56] Query camera_b_instant: 1 violation(s) written — upserted: 1, modified: 0
[Batch 20] Query avg_bc: 2 violation(s) written — upserted: 0, modified: 2
[Batch 53] Query camera_c_instant: 1 violation(s) written — upserted: 1, modified: 0
[Batch 56] Query camera_a_instant: 9 violation(s) written — upserted: 9, modified: 0
Batch 57 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 54] Query camera_c_instant: 1 violation(s) written — upserted: 1, modified: 0
[Batch 57] Query camera_a_instant: 7 violation(s) written — upserted: 7, modified: 0
[Batch 58] Query camera_b_instant: 2 violation(s) written — upserted: 0, modified: 2
[Batch 21] Query avg_ab: 4 violation(s) written — upserted: 0, modified: 4
Batch 55 is empty, skipping MongoDB write. resolving: camera_c_instant
[Batch 58] Query camera_a_in

[Batch 29] Query avg_bc: 1 violation(s) written — upserted: 0, modified: 1
Batch 85 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 81] Query camera_c_instant: 2 violation(s) written — upserted: 0, modified: 2
[Batch 84] Query camera_a_instant: 12 violation(s) written — upserted: 12, modified: 0
[Batch 85] Query camera_a_instant: 10 violation(s) written — upserted: 10, modified: 0
Batch 82 is empty, skipping MongoDB write. resolving: camera_c_instant
[Batch 86] Query camera_b_instant: 3 violation(s) written — upserted: 0, modified: 3
[Batch 30] Query avg_ab: 5 violation(s) written — upserted: 0, modified: 5
[Batch 83] Query camera_c_instant: 2 violation(s) written — upserted: 0, modified: 2
[Batch 87] Query camera_b_instant: 1 violation(s) written — upserted: 0, modified: 1
[Batch 86] Query camera_a_instant: 7 violation(s) written — upserted: 7, modified: 0
[Batch 84] Query camera_c_instant: 1 violation(s) written — upserted: 0, modified: 1
Batch 30 is empty, skipp

[Batch 109] Query camera_a_instant: 11 violation(s) written — upserted: 11, modified: 0
[Batch 111] Query camera_c_instant: 2 violation(s) written — upserted: 0, modified: 2
Batch 113 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 110] Query camera_a_instant: 9 violation(s) written — upserted: 9, modified: 0
[Batch 40] Query avg_ab: 4 violation(s) written — upserted: 0, modified: 4
[Batch 40] Query avg_bc: 4 violation(s) written — upserted: 0, modified: 4
[Batch 112] Query camera_c_instant: 1 violation(s) written — upserted: 1, modified: 0
Batch 114 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 111] Query camera_a_instant: 8 violation(s) written — upserted: 8, modified: 0
Batch 113 is empty, skipping MongoDB write. resolving: camera_c_instant
Batch 115 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 112] Query camera_a_instant: 9 violation(s) written — upserted: 9, modified: 0
Batch 41 is empty, skipping MongoDB write. r

...

Batch 155 is empty, skipping MongoDB write. resolving: avg_bc
Batch 378 is empty, skipping MongoDB write. resolving: camera_c_instant
[Batch 158] Query avg_ab: 3 violation(s) written — upserted: 0, modified: 3
Batch 381 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 375] Query camera_a_instant: 10 violation(s) written — upserted: 10, modified: 0
[Batch 379] Query camera_c_instant: 1 violation(s) written — upserted: 0, modified: 1
[Batch 382] Query camera_b_instant: 2 violation(s) written — upserted: 0, modified: 2
[Batch 376] Query camera_a_instant: 8 violation(s) written — upserted: 8, modified: 0
[Batch 156] Query avg_bc: 1 violation(s) written — upserted: 0, modified: 1
Batch 380 is empty, skipping MongoDB write. resolving: camera_c_instant
[Batch 159] Query avg_ab: 3 violation(s) written — upserted: 0, modified: 3
[Batch 377] Query camera_a_instant: 8 violation(s) written — upserted: 8, modified: 0
[Batch 383] Query camera_b_instant: 2 violation(s) written — u

Batch 404 is empty, skipping MongoDB write. resolving: camera_c_instant
[Batch 407] Query camera_b_instant: 1 violation(s) written — upserted: 0, modified: 1
[Batch 401] Query camera_a_instant: 11 violation(s) written — upserted: 11, modified: 0
Batch 168 is empty, skipping MongoDB write. resolving: avg_bc
[Batch 405] Query camera_c_instant: 1 violation(s) written — upserted: 0, modified: 1
Batch 408 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 402] Query camera_a_instant: 9 violation(s) written — upserted: 9, modified: 0
[Batch 172] Query avg_ab: 2 violation(s) written — upserted: 0, modified: 2
[Batch 406] Query camera_c_instant: 1 violation(s) written — upserted: 0, modified: 1
Batch 409 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 403] Query camera_a_instant: 8 violation(s) written — upserted: 8, modified: 0
[Batch 407] Query camera_c_instant: 1 violation(s) written — upserted: 0, modified: 1
[Batch 169] Query avg_bc: 1 violation(s) w

[Batch 427] Query camera_a_instant: 9 violation(s) written — upserted: 9, modified: 0
Batch 183 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 431] Query camera_c_instant: 1 violation(s) written — upserted: 0, modified: 1
Batch 434 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 180] Query avg_bc: 1 violation(s) written — upserted: 0, modified: 1
[Batch 428] Query camera_a_instant: 7 violation(s) written — upserted: 7, modified: 0
[Batch 432] Query camera_c_instant: 1 violation(s) written — upserted: 0, modified: 1
Batch 435 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 184] Query avg_ab: 3 violation(s) written — upserted: 0, modified: 3
[Batch 429] Query camera_a_instant: 10 violation(s) written — upserted: 10, modified: 0
[Batch 433] Query camera_c_instant: 2 violation(s) written — upserted: 2, modified: 0
[Batch 181] Query avg_bc: 2 violation(s) written — upserted: 0, modified: 2
[Batch 436] Query camera_b_instant: 2 violation(

[Batch 453] Query camera_a_instant: 9 violation(s) written — upserted: 9, modified: 0
[Batch 457] Query camera_c_instant: 2 violation(s) written — upserted: 2, modified: 0
[Batch 460] Query camera_b_instant: 1 violation(s) written — upserted: 0, modified: 1
Batch 196 is empty, skipping MongoDB write. resolving: avg_ab
Batch 193 is empty, skipping MongoDB write. resolving: avg_bc
[Batch 454] Query camera_a_instant: 16 violation(s) written — upserted: 15, modified: 1
Batch 458 is empty, skipping MongoDB write. resolving: camera_c_instant
Batch 461 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 455] Query camera_a_instant: 13 violation(s) written — upserted: 13, modified: 0
[Batch 459] Query camera_c_instant: 2 violation(s) written — upserted: 0, modified: 2
Batch 462 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 197] Query avg_ab: 2 violation(s) written — upserted: 0, modified: 2
[Batch 456] Query camera_a_instant: 9 violation(s) written — ups

Batch 486 is empty, skipping MongoDB write. resolving: camera_b_instant
[Batch 208] Query avg_ab: 3 violation(s) written — upserted: 0, modified: 3
[Batch 484] Query camera_c_instant: 1 violation(s) written — upserted: 0, modified: 1
[Batch 480] Query camera_a_instant: 11 violation(s) written — upserted: 10, modified: 1
[Batch 205] Query avg_bc: 2 violation(s) written — upserted: 0, modified: 2
[Batch 487] Query camera_b_instant: 2 violation(s) written — upserted: 0, modified: 2
Batch 485 is empty, skipping MongoDB write. resolving: camera_c_instant
[Batch 481] Query camera_a_instant: 12 violation(s) written — upserted: 12, modified: 0
[Batch 488] Query camera_b_instant: 3 violation(s) written — upserted: 1, modified: 2
[Batch 209] Query avg_ab: 2 violation(s) written — upserted: 0, modified: 2
[Batch 486] Query camera_c_instant: 1 violation(s) written — upserted: 1, modified: 0
[Batch 482] Query camera_a_instant: 13 violation(s) written — upserted: 12, modified: 1
[Batch 206] Query av